## # Silver Base Transformation – Geography

This notebook prepares the base geography dataset used by multiple Silver transformations.

**Source dataset:**
capstone.bronze.geography_population_raw

**Transformation logic:**
- Removes header rows and invalid records
- Keeps only county-level GEO_ID values
- Extracts state_fips and county_fips
- Creates location_id by combining state and county FIPS codes
- Standardizes county and state names

**Output:**
Creates a reusable dataframe geo that will be used by other notebooks.

**Note:**
This notebook does not write a table.
It is executed using %run in downstream notebooks such as location_county and population.

In [0]:
%python
from pyspark.sql import functions as F

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Source bronze table
src = spark.table("capstone.bronze.geography_population_raw")

# Base geo dataframe (county-level only, cleaned)
geo = (
  src
  .filter(F.col("GEO_ID").isNotNull())
  .filter(F.col("GEO_ID") != "Geography")
  .filter(F.col("GEO_ID").rlike(r"^0500000US\d{5}$"))  # county-level only
  .withColumn("state_fips", F.regexp_extract("GEO_ID", r"^0500000US(\d{2})\d{3}$", 1))
  .withColumn("county_fips", F.regexp_extract("GEO_ID", r"^0500000US\d{2}(\d{3})$", 1))
  .withColumn("location_id", F.concat(F.col("state_fips"), F.col("county_fips")))
  .withColumn("county_name", F.trim(F.regexp_extract("NAME", r"^(.*?),", 1)))
  .withColumn("state_name", F.trim(F.regexp_extract("NAME", r",\s*(.*)$", 1)))
  .withColumn("county_name", F.regexp_replace(F.col("county_name"), r"\s+", " "))
  .withColumn("state_name", F.regexp_replace(F.col("state_name"), r"\s+", " "))
  .filter(
    (F.length("state_fips") == 2) &
    (F.length("county_fips") == 3) &
    (F.length("location_id") == 5)
  )
)

print("Base geography ready: geo dataframe created")